# 🚀 Colab Turbo Downloader

Fill settings → Runtime → Run all


In [ ]:
#@title 🔧 Settings
URL = ""  #@param {type:"string"}
NAME = "file.bin"  #@param {type:"string"}
THREADS = 6  #@param {type:"integer"}

print("✅ Settings ready")


In [ ]:
#@title 🔗 Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")


In [ ]:
#@title 🚀 Download (Live Progress)
import requests, math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

dest = Path("/content/drive/MyDrive/Downloads")
dest.mkdir(exist_ok=True)
file = dest / NAME

def get_size(url):
    try:
        return int(requests.head(url).headers.get("content-length",0))
    except:
        return 0

size = get_size(URL)
pbar = tqdm(total=size if size else None, unit='B', unit_scale=True)

def download_part(i, start, end):
    headers = {"Range": f"bytes={start}-{end}"} if end else {}
    with requests.get(URL, headers=headers, stream=True) as r:
        with open(dest/f"part_{i}", "wb") as f:
            for chunk in r.iter_content(1024*512):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

if size:
    part_size = math.ceil(size / THREADS)
    with ThreadPoolExecutor(max_workers=THREADS) as ex:
        ex.map(lambda i: download_part(i, i*part_size, min(size-1,(i+1)*part_size-1)), range(THREADS))

    with open(file, "wb") as out:
        for i in range(THREADS):
            part = dest/f"part_{i}"
            if part.exists():
                out.write(open(part,"rb").read())
                part.unlink()
else:
    download_part(0,0,None)

pbar.close()
print("✅ Download complete:", file)
